# 使用線性迴歸預測波士頓房價

**本分析的目標是使用這份資料建立線性迴歸模型，預測房屋價格。**

資料包含以下欄位：
* `crim`：各城鎮的人均犯罪率。
* `zn`：劃為面積超過 25,000 平方英尺住宅用地的比例。
* `indus`：各城鎮非零售業用地的比例。
* `chas`：查爾斯河虛擬變數（地區與河流相鄰為 1，否則為 0）。
* `nox`：氮氧化物濃度（每一千萬份中的含量）。
* `rm`：每戶住宅的平均房間數。
* `age`：屋主自住住宅中，於 1940 年以前建成的比例。
* `dis`：到波士頓五個就業中心距離的加權平均值。
* `rad`：放射狀公路的交通可達性指標。
* `tax`：每 10,000 美元完整估值所課徵的房產稅率。
* `ptratio`：各城鎮的學生與教師人數比。
* `black`：1000(Bk - 0.63)^2，其中 Bk 為各城鎮黑人居民的比例。
* `lstat`：社會地位較低人口的百分比。
* `medv`：屋主自住住宅價值的中位數（單位：千美元）。

附註：這是我的第一次分析，我正在學習如何解讀圖表。

**開始分析**

首先匯入所需的套件，準備分析環境。

In [49]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn import metrics
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

%matplotlib inline

In [50]:
# 匯入資料集並查看資料
BostonTrain = pd.read_csv("../input/boston_train.csv")

**在這裡查看 BostonTrain 的資料。**

In [51]:
BostonTrain.head()

In [52]:
BostonTrain.info()
BostonTrain.describe()

**接下來要思考各欄位的意義，找出適合用來建立模型的欄位。如果納入與預測目標 `medv` 無關的欄位，模型的效果可能不佳。**

In [53]:
# ID 欄位與本次分析無關。
BostonTrain.drop('ID', axis = 1, inplace=True)

In [54]:
BostonTrain.plot.scatter('rm', 'medv')

從這張散佈圖可以看出明顯的線性趨勢：每戶住宅的平均房間數越多，住宅價值的中位數通常越高。

**接著觀察所有變數彼此之間的關係。**

In [55]:
plt.subplots(figsize=(12,8))
sns.heatmap(BostonTrain.corr(), cmap = 'RdGy')

相較於成對關係圖（pairplot），這張熱圖更方便觀察各變數的相關性。

請聚焦在最後一列，也就是縱軸為 `medv` 的位置：

* 紅色／橘色：顏色越紅，代表該變數與 `medv` 的負相關越強；該變數越大，`medv` 通常越小。
* 淺色：代表橫軸與縱軸變數的線性相關性接近零。
* 灰色／黑色：顏色越黑，代表該變數與 `medv` 的正相關越強；該變數越大，`medv` 通常越大。

**繪製成對關係圖，觀察不同變數之間的相關性。**

負相關。

當 x 較高時，y 通常較低，反之亦然。

越往右，負相關越弱。

In [56]:
sns.pairplot(BostonTrain, vars = ['lstat', 'ptratio', 'indus', 'tax', 'crim', 'nox', 'rad', 'age', 'medv'])

零相關：x 與 y 之間沒有線性相關。

正相關：x 與 y 通常同向變動。

越往右，相關性越弱。

In [57]:
sns.pairplot(BostonTrain, vars = ['rm', 'zn', 'black', 'dis', 'chas','medv'])

# 訓練線性迴歸模型

**定義 X 與 y**

* X：預測變數，也稱為自變數或特徵。
* y：反應變數，也稱為依變數。

In [58]:
X = BostonTrain[['crim', 'zn', 'indus', 'chas', 'nox', 'rm', 'age', 'dis', 'rad', 'tax',
       'ptratio', 'black', 'lstat']]
y = BostonTrain['medv']

**匯入 sklearn 套件中的工具：**

* `train_test_split`：將資料分成兩組，一組用來建立模型，另一組用來驗證模型。
* `LinearRegression`：用來建立線性迴歸模型。

In [60]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4)

In [61]:
lm = LinearRegression()
lm.fit(X_train,y_train)

In [62]:
predictions = lm.predict(X_test)

In [63]:
plt.scatter(y_test,predictions)
plt.xlabel('Y Test')
plt.ylabel('Predicted Y')

In [64]:
print('MAE:', metrics.mean_absolute_error(y_test, predictions))
print('MSE:', metrics.mean_squared_error(y_test, predictions))
print('RMSE:', np.sqrt(metrics.mean_squared_error(y_test, predictions)))

以均方根誤差（RMSE）評估模型的預測誤差。由於 `medv` 的單位是千美元，將 RMSE 乘以 1,000，即可換算為美元。

In [65]:
sns.distplot((y_test-predictions),bins=50);

殘差的分布越接近常態分布越理想。

In [66]:
coefficients = pd.DataFrame(lm.coef_,X.columns)
coefficients.columns = ['coefficients']
coefficients

如何解讀這些迴歸係數：

係數表示在其他變數保持不變時，某個變數每增加一個單位，預測的 `medv` 會改變多少。

* `nox` 的係數若為負，`nox` 每增加一個單位，預測房價就會減少該係數絕對值乘以 1,000 美元。
* `rm` 的係數若為正，`rm` 每增加一個單位，預測房價就會增加該係數乘以 1,000 美元。

乘以 1,000 是因為 `medv` 的單位為千美元。其他變數的係數也可以用相同方式解讀。

如前面所說，這是我第一次使用機器學習方法進行分析。希望未來能持續進步，英文能力也能一起提升。=)

謝謝！